In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split, GridSearchCV
from tqdm import tqdm
import joblib
import category_encoders as ce  # Target Encoding
from contextlib import contextmanager


# ---------------------------
# Utility Function: tqdm_joblib for Progress Tracking
# ---------------------------
@contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to add progress bar to GridSearchCV."""

    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


# ---------------------------
# Feature Engineering: Adding Nonlinear and Target Encoded Features
# ---------------------------
def feature_engineering(df):
    """Enhance dataset with polynomial and categorical features."""
    df = df.copy()

    # Compute Absolute Seed Difference
    df["Abs_SeedDiff"] = df["SeedDiff"].abs()

    # Polynomial Features
    df["SeedValue1_sq"] = df["SeedValue1"] ** 2
    df["SeedValue2_sq"] = df["SeedValue2"] ** 2
    df["SeedProduct"] = df["SeedValue1"] * df["SeedValue2"]

    # Target Encoding for TeamID
    encoder = ce.TargetEncoder(cols=["TeamID1", "TeamID2"])
    df[["TeamID1_encoded", "TeamID2_encoded"]] = encoder.fit_transform(
        df[["TeamID1", "TeamID2"]], df["Outcome"]
    )

    return df


# ---------------------------
# XGBoost Model Training with Hyperparameter Tuning
# ---------------------------
def train_xgboost_with_tuning(model_data, gender):
    """Train an optimized XGBoost model on SeedDiff and engineered features."""
    for season in range(2021, 2025):  # Predicting for 2021-2024
        train_data = model_data[model_data["Season"] < season].copy()
        test_data = model_data[model_data["Season"] == season].copy()

        # Apply feature engineering
        train_data = feature_engineering(train_data)
        test_data = feature_engineering(test_data)

        # Define features to use in training
        features = [
            "SeedValue1",
            "SeedValue2",
            "SeedDiff",
            "Abs_SeedDiff",
            "SeedValue1_sq",
            "SeedValue2_sq",
            "SeedProduct",
            "TeamID1_encoded",
            "TeamID2_encoded",
        ]

        X_train, y_train = train_data[features], train_data["Outcome"]
        X_test, y_test = test_data[features], test_data["Outcome"]

        # Define XGBoost Model
        xgb_model = xgb.XGBClassifier(eval_metric="logloss", use_label_encoder=False)

        # Hyperparameter Grid
        param_grid = {
            "n_estimators": [100, 300, 500],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 5, 7],
            "reg_lambda": [0.1, 1, 10],  # L2 Regularization
        }

        # Grid Search with Progress Bar
        grid_search = GridSearchCV(
            xgb_model, param_grid, cv=5, scoring="neg_log_loss", verbose=0, n_jobs=-1
        )
        total_iter = (
            len(param_grid["n_estimators"])
            * len(param_grid["learning_rate"])
            * len(param_grid["max_depth"])
            * len(param_grid["reg_lambda"])
            * 5
        )  # Candidates * 5-fold CV

        with tqdm_joblib(tqdm(desc="XGBoost Grid Search", total=total_iter)):
            grid_search.fit(X_train, y_train)

        # Best Model from Grid Search
        best_xgb = grid_search.best_estimator_
        y_pred_probs = best_xgb.predict_proba(X_test)[:, 1]

        # Compute Metrics
        log_loss_value = log_loss(y_test, y_pred_probs)
        brier_score = brier_score_loss(y_test, y_pred_probs)
        auc_value = roc_auc_score(y_test, y_pred_probs)

        # Log to MLflow
        with mlflow.start_run(run_name=f"{gender}_XGBoost_Tuned_{season}"):
            mlflow.log_params(grid_search.best_params_)
            mlflow.log_param("Train_Size", len(X_train))
            mlflow.log_metric("Log_Loss", log_loss_value)
            mlflow.log_metric("Brier_Score", brier_score)
            mlflow.log_metric("AUC", auc_value)
            mlflow.xgboost.log_model(best_xgb, f"{gender}_XGBoost_Tuned_Model_{season}")

        print(
            f"{gender} XGBoost Tuned Model for Season {season} - Log Loss: {log_loss_value:.4f}, AUC: {auc_value:.4f}, Brier Score: {brier_score:.4f}"
        )

In [ ]:
# ---------------------------
# Train Men's and Women's Models with XGBoost & Hyperparameter Tuning
# ---------------------------
train_xgboost_with_tuning(m_model_data, "Men")
train_xgboost_with_tuning(w_model_data, "Women")